In [1]:
!pip install -q pymupdf groq transformers peft trl bitsandbytes accelerate datasets sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 52.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible

In [2]:
import json
import random
import re

random.seed(42)

with open("/kaggle/input/datasets/tomatopdf/single-cell-qa-dataset/instruction_dataset.jsonl") as f:
    dataset = [json.loads(line) for line in f]

random.shuffle(dataset)

split = int(len(dataset) * 0.9)
train_data = dataset[:split]
val_data   = dataset[split:]

def format_example(item):
    def clean_text(text):
        text = text.replace("in the passage", "in this domain")
        text = text.replace("mentioned in the passage", "mentioned")
        text = text.replace("described in the passage", "described")
        text = text.replace("discussed in the passage", "discussed")
        text = text.replace("from the passage", "")
        text = text.replace("the passage", "the literature")
        text = text.replace("The passage", "The literature")
        text = re.sub(r'\s+\?', '?', text)
        text = re.sub(r'\s+\.', '.', text)
        text = re.sub(r'\s{2,}', ' ', text)
        return text.strip()

    instruction = clean_text(item['instruction'])
    response    = clean_text(item['response'])

    return {
        "text": (
            f"<|begin_of_text|>"
            f"<|start_header_id|>system<|end_header_id|>\n"
            f"You are a helpful scientific assistant specializing in "
            f"single-cell transcriptomics and computational genomics.\n"
            f"<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n"
            f"{instruction}\n"
            f"<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
            f"{response}"
            f"<|eot_id|>"
        )
    }


train_formatted = [format_example(i) for i in train_data]
val_formatted   = [format_example(i) for i in val_data]

with open("/kaggle/working/train.jsonl", "w") as f:
    for item in train_formatted:
        f.write(json.dumps(item) + "\n")

with open("/kaggle/working/val.jsonl", "w") as f:
    for item in val_formatted:
        f.write(json.dumps(item) + "\n")

print(f"Train: {len(train_formatted)} examples")
print(f"Val:   {len(val_formatted)} examples")
print(f"\nSample:")
print(train_formatted[0]["text"][:600])

Train: 1179 examples
Val:   132 examples

Sample:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful scientific assistant specializing in single-cell transcriptomics and computational genomics.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What is the location of the KDD conference mentioned in this domain?
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
The KDD conference mentioned in this domain is scheduled to take place on Jeju Island, Republic of Korea.<|eot_id|>


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import gc
gc.collect()
torch.cuda.empty_cache()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count visible: {torch.cuda.device_count()}")
print(f"GPU 0: {torch.cuda.get_device_name(0)}")

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    token=hf_token,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU count visible: 1
GPU 0: Tesla T4
Loading tokenizer...


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded. Parameters: 1,235,814,400


In [4]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# Force all embedding layers to cuda:0
for name, param in model.named_parameters():
    if "embed" in name:
        param.data = param.data.to("cuda:0")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [5]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [6]:
from transformers.utils import logging

logging.set_verbosity_info()
logging.enable_default_handler()
logging.enable_explicit_format()

# Disable notebook widgets
os.environ["DISABLE_TQDM"] = "1"

In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/working/train.jsonl",
        "validation": "/kaggle/working/val.jsonl"
    }
)

print(f"Train: {len(dataset['train'])}  Val: {len(dataset['validation'])}")

sft_config = SFTConfig(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_steps=60,
    bf16=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataset_text_field="text",
    disable_tqdm=True,
    logging_strategy="steps",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete.")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

[INFO|training_args.py:1741] 2026-06-29 08:03:32,023 >> PyTorch: setting up devices


Train: 1179  Val: 132


Adding EOS to train dataset:   0%|          | 0/1179 [00:00<?, ? examples/s]

...

{'loss': '0.08977', 'grad_norm': '0.8547', 'learning_rate': '3.671e-10', 'entropy': '0.1197', 'num_tokens': '1.364e+06', 'mean_token_accuracy': '0.9643', 'epoch': '10'}


[INFO|trainer.py:4115] 2026-06-29 10:44:53,077 >> Saving model checkpoint to /kaggle/working/checkpoints/checkpoint-1480


{'eval_loss': '2.046', 'eval_runtime': '27.46', 'eval_samples_per_second': '4.806', 'eval_steps_per_second': '2.403', 'eval_entropy': '0.5246', 'eval_num_tokens': '1.364e+06', 'eval_mean_token_accuracy': '0.7174', 'epoch': '10'}


[INFO|configuration_utils.py:667] 2026-06-29 10:44:53,550 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6/config.json
[INFO|configuration_utils.py:739] 2026-06-29 10:44:53,551 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_fac

{'train_runtime': '9680', 'train_samples_per_second': '1.218', 'train_steps_per_second': '0.153', 'train_loss': '0.4999', 'epoch': '10'}
Training complete.


In [8]:
from peft import PeftModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

import torch

user_secrets = UserSecretsClient()
hf_write_token = user_secrets.get_secret("HF_WRITE_TOKEN")
login(token=hf_write_token)

# Save the LoRA adapter
adapter_path = "/kaggle/working/lora_adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

# Merge LoRA weights back into base model
# Need to reload base model in full precision for merge
print("Reloading base model for merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()

merged_path = "/kaggle/working/merged_model"
merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

repo_id = "coconutpdf/genomics-llama-1b"

merged_model.push_to_hub(
    repo_id,
    private=False,      # True if you want a private model
)

tokenizer.push_to_hub(repo_id)

print("Model uploaded!")

print(f"Merged model saved to {merged_path}")

[INFO|configuration_utils.py:667] 2026-06-29 10:44:54,977 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6/config.json
[INFO|configuration_utils.py:739] 2026-06-29 10:44:54,978 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_fac

Adapter saved to /kaggle/working/lora_adapter
Reloading base model for merge...


[INFO|configuration_utils.py:667] 2026-06-29 10:44:55,463 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6/config.json
[INFO|configuration_utils.py:739] 2026-06-29 10:44:55,464 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "float16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_fact

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[INFO|configuration_utils.py:967] 2026-06-29 10:44:57,687 >> loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6/generation_config.json
[INFO|configuration_utils.py:1014] 2026-06-29 10:44:57,687 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "temperature": 0.6,
  "top_p": 0.9
}

[INFO|configuration_utils.py:432] 2026-06-29 10:44:58,026 >> Configuration saved in /kaggle/working/merged_model/config.json
[INFO|configuration_utils.py:803] 2026-06-29 10:44:58,027 >> Configuration saved in /kaggle/working/merged_model/generation_config.json


...

[INFO|tokenization_utils_base.py:3327] 2026-06-29 10:46:01,237 >> chat template saved in /tmp/tmpkdrycgwo/chat_template.jinja
[INFO|tokenization_utils_base.py:2181] 2026-06-29 10:46:01,239 >> tokenizer config file saved in /tmp/tmpkdrycgwo/tokenizer_config.json
[INFO|hub.py:700] 2026-06-29 10:46:01,457 >> Uploading the following files to coconutpdf/genomics-llama-1b: tokenizer.json,tokenizer_config.json,chat_template.jinja,README.md


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Model uploaded!
Merged model saved to /kaggle/working/merged_model


In [9]:
# Test the merged model directly in transformers
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    device_map="auto",
)

test_passage = """
DNA methylation is an epigenetic mechanism involving the addition of 
a methyl group to the cytosine base of DNA, primarily at CpG dinucleotides. 
This modification plays a crucial role in gene regulation, 
genomic imprinting, and the silencing of transposable elements. 
Aberrant DNA methylation patterns are commonly observed in cancer cells,
where tumor suppressor genes are often hypermethylated and silenced.
"""

test_prompt = (
    f"<|begin_of_text|>"
    f"<|start_header_id|>system<|end_header_id|>\n"
    f"You are a helpful scientific assistant. "
    f"Answer questions based only on the provided passage.\n"
    f"<|eot_id|>"
    f"<|start_header_id|>user<|end_header_id|>\n"
    f"Passage: {test_passage}\n\n"
    f"Question: What role does DNA methylation play in cancer?\n"
    f"<|eot_id|>"
    f"<|start_header_id|>assistant<|end_header_id|>\n"
)

result = pipe(test_prompt)
print(result[0]["generated_text"][len(test_prompt):])

[WARNING|logging.py:327] 2026-06-29 10:46:05,361 >> Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[WARNING|utils.py:1723] 2026-06-29 10:46:05,366 >> Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DNA methylation plays a crucial role in cancer by silencing tumor suppressor genes, which are often hypermethylated and silenced.


# loading from huggingface

In [10]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import torch

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

repo_id= 'coconutpdf/genomics-llama-1b'
tokenizer = AutoTokenizer.from_pretrained(repo_id)

model = AutoModelForCausalLM.from_pretrained(repo_id,torch_dtype="auto",device_map="auto")

config.json:   0%|          | 0.00/893 [00:00<?, ?B/s]

[INFO|configuration_utils.py:667] 2026-06-29 10:46:07,439 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--coconutpdf--genomics-llama-1b/snapshots/df6c68e4f65856fa3a9d1c634f5e4eb1a3a1406f/config.json
[INFO|configuration_utils.py:739] 2026-06-29 10:46:07,441 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "float16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_factor":

tokenizer_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

[INFO|modeling_utils.py:732] 2026-06-29 10:46:52,225 >> loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--coconutpdf--genomics-llama-1b/snapshots/df6c68e4f65856fa3a9d1c634f5e4eb1a3a1406f/model.safetensors
[INFO|modeling_utils.py:801] 2026-06-29 10:46:52,226 >> Will use dtype=torch.float16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-06-29 10:46:52,228 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-06-29 10:46:52,253 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

[INFO|configuration_utils.py:967] 2026-06-29 10:46:53,529 >> loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--coconutpdf--genomics-llama-1b/snapshots/df6c68e4f65856fa3a9d1c634f5e4eb1a3a1406f/generation_config.json
[INFO|configuration_utils.py:1014] 2026-06-29 10:46:53,530 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "temperature": 0.6,
  "top_p": 0.9
}



In [12]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="coconutpdf/genomics-llama-1b",
    device_map="auto"
)

[INFO|configuration_utils.py:667] 2026-06-29 10:46:53,829 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--coconutpdf--genomics-llama-1b/snapshots/df6c68e4f65856fa3a9d1c634f5e4eb1a3a1406f/config.json
[INFO|configuration_utils.py:739] 2026-06-29 10:46:53,830 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "float16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_factor":

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[INFO|configuration_utils.py:967] 2026-06-29 10:46:55,235 >> loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--coconutpdf--genomics-llama-1b/snapshots/df6c68e4f65856fa3a9d1c634f5e4eb1a3a1406f/generation_config.json
[INFO|configuration_utils.py:1014] 2026-06-29 10:46:55,236 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "temperature": 0.6,
  "top_p": 0.9
}

[INFO|configuration_utils.py:667] 2026-06-29 10:46:55,237 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--coconutpdf--genomics-llama-1b/snapshots/df6c68e4f65856fa3a9d1c634f5e4eb1a3a1406f/config.json
[INFO|configuration_utils.py:739] 2026-06-29 10:46:55,239 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "float16",
  "eos

In [13]:
test_prompt = (
    f"<|begin_of_text|>"
    f"<|start_header_id|>system<|end_header_id|>\n"
    f"You are a helpful scientific assistant specializing in "
    f"single-cell transcriptomics and computational genomics. "
    f"Answer questions based only on the provided passage.\n"
    f"<|eot_id|>"
    f"<|start_header_id|>user<|end_header_id|>\n"
    f"How was the hierarchical graph transformer model HEIST evaluated?.\n"
    f"<|eot_id|>"
    f"<|start_header_id|>assistant<|end_header_id|>\n"
)

result = pipe(test_prompt, max_new_tokens=600)
print(result[0]["generated_text"][len(test_prompt):])

[WARNING|utils.py:1723] 2026-06-29 10:46:57,926 >> Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The hierarchical graph transformer model HEIST was evaluated on 30k cells with 5,000 random gene perturbations. The model was trained with a two-stage setup: first, the full evaluation set was fine-tuned to zero. Then, the full evaluation set was split into training and validation sets. The model was evaluated on the full evaluation set in both the original and zero-shot settings.
